<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/m7_0_m7_1_colab_diagnostic_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M7.0 & M7.1 — DynaTree-CRE Gate 5 Diagnostic & TACRED `DT-C0` Control (Google Colab GPU)
- **M7.0:** Chạy `DT-C0` TACRED (`t1_centered_cosine` & `raw_cosine`) + kiểm tra **tensor equality & prediction equality** trên checkpoint `.pt` ở Google Drive.
- **M7.1:** Phân rã lỗi định tuyến 2×2, `Recall@1/3/5/10`, per-cluster/singleton, calibration trên checkpoint `DT-C1..DT-C4` hiện có.
- **M7.2 / M7.3 Zero-Retrain Diagnostic:** Thử `raw_cosine` retrieval và `size_corrected_hierarchical` / `hybrid_tree_routing` trên checkpoint hiện có trước khi retrain.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chon GPU runtime (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/dynatree-cre'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
gdrive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path('/content/research-code-lab/runs/dynatree_cre')
local_runs.parent.mkdir(parents=True, exist_ok=True)
if local_runs.is_symlink():
    local_runs.unlink()
local_runs.symlink_to(gdrive_runs)
print('Linked runs ->', gdrive_runs)

In [ ]:
# M7.0: Chay DT-C0 TACRED (t1_centered_cosine & raw_cosine)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_tacred_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_tacred_5shot_seed2021.yaml \
    --similarity-space raw_cosine --threshold-mode fixed --insertion-threshold 0.75 \
    --output-dir runs/dynatree_cre/dt_c0_raw_cosine/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

In [ ]:
# M7.0 + M7.1: Diagnostic tren cac checkpoint .pt co san trong Google Drive
diag_dir = Path('/content/research-code-lab/runs/dynatree_cre/m7_diagnostics')
diag_dir.mkdir(parents=True, exist_ok=True)

targets = [
    ('dt_c1_fewrel', 'runs/dynatree_cre/dt_c1_flat_hard/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c2_fewrel', 'runs/dynatree_cre/dt_c2_flat_soft/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c3_fewrel', 'runs/dynatree_cre/dt_c3_cluster_flat_route/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c4_fewrel', 'runs/dynatree_cre/dt_c4_hier_soft/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c4_tacred', 'runs/dynatree_cre/dt_c4_hier_soft/tacred_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c2_expB', 'runs/dynatree_cre/dt_c2_semantic_homogeneous/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c4_expB', 'runs/dynatree_cre/dt_c4_semantic_homogeneous/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c2_expC', 'runs/dynatree_cre/dt_c2_cross_task_dispersed/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
    ('dt_c4_expC', 'runs/dynatree_cre/dt_c4_cross_task_dispersed/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'),
]

# 1. Resume tensor & prediction equality check (DT-C4 FewRel)
c4_main = 'runs/dynatree_cre/dt_c4_hier_soft/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'
c4_res = 'runs/dynatree_cre/dt_c4_resume_check/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt'
if Path(c4_main).exists() and Path(c4_res).exists():
    !python experiments/diagnose_dynatree_checkpoint.py \
        --checkpoint {c4_main} --compare-checkpoint {c4_res} \
        --device cuda --allow-hub-download \
        --output-json {diag_dir / 'dt_c4_fewrel_resume_equality.json'}

# 2. M7.1 decomposition cho tung checkpoint
for tag, ckpt in targets:
    if Path(ckpt).exists():
        !python experiments/diagnose_dynatree_checkpoint.py \
            --checkpoint {ckpt} --device cuda --allow-hub-download \
            --output-json {diag_dir / f'{tag}_m7_1.json'}

# 3. M7.2 & M7.3 zero-retrain inference diagnostic tren DT-C4 FewRel & Exp B
for tag, ckpt in [('dt_c4_fewrel', c4_main), ('dt_c4_expB', 'runs/dynatree_cre/dt_c4_semantic_homogeneous/fewrel_5shot/seed_2021/checkpoints/after_T8/state.pt')]:
    if Path(ckpt).exists():
        !python experiments/diagnose_dynatree_checkpoint.py \
            --checkpoint {ckpt} --use-raw-retrieval --device cuda --allow-hub-download \
            --output-json {diag_dir / f'{tag}_m7_2_raw_retrieval.json'}
        !python experiments/diagnose_dynatree_checkpoint.py \
            --checkpoint {ckpt} --routing-override size_corrected_hierarchical --eta 1.0 --device cuda --allow-hub-download \
            --output-json {diag_dir / f'{tag}_m7_3_eta1.json'}
        !python experiments/diagnose_dynatree_checkpoint.py \
            --checkpoint {ckpt} --routing-override hybrid_tree_routing --gamma 0.5 --device cuda --allow-hub-download \
            --output-json {diag_dir / f'{tag}_m7_3_hybrid_g05.json'}

In [ ]:
import shutil
from google.colab import files

bundle = Path('/content/m7_diagnostics_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)
shutil.copytree('runs/dynatree_cre/m7_diagnostics', bundle / 'm7_diagnostics', dirs_exist_ok=True)
for tac in ['dt_c0_prototype_control/tacred_5shot/seed_2021', 'dt_c0_raw_cosine/tacred_5shot/seed_2021']:
    p = Path('runs/dynatree_cre') / tac
    if p.exists():
        shutil.copytree(p, bundle / tac, ignore=shutil.ignore_patterns('*.pt'), dirs_exist_ok=True)
shutil.make_archive('/content/m7_diagnostics_bundle', 'zip', bundle)
files.download('/content/m7_diagnostics_bundle.zip')